In [32]:
import pandas as pd

df_train = pd.read_csv('/content/drive/MyDrive/CienciaDeDatos/Viajes_train.csv')
df_test = pd.read_csv('/content/drive/MyDrive/CienciaDeDatos/Viajes_test.csv')

print(df_train.shape, df_test.shape)
df_train.head()

(3512, 9) (879, 9)


,hora,dia_semana,mes,es_fin_de_semana,hora_sin,hora_cos,es_festivo,tipo_zona_encoded,label
0,15,4,7,0,-0.707107,-7.071068e-01,0,0,85
1,6,2,4,0,1.000000,6.123234e-17,0,0,26
2,13,5,7,1,-0.258819,-9.659258e-01,0,0,27
3,21,0,8,0,-0.707107,7.071068e-01,0,0,61
4,10,3,8,0,0.500000,-8.660254e-01,0,0,40


In [33]:
df_test

,hora,dia_semana,mes,es_fin_de_semana,hora_sin,hora_cos,es_festivo,tipo_zona_encoded,label
0,2,2,9,0,0.500000,8.660254e-01,0,0,5
1,3,0,4,0,0.707107,7.071068e-01,0,0,2
2,21,3,8,0,-0.707107,7.071068e-01,0,0,142
3,6,1,8,0,1.000000,6.123234e-17,0,0,29
4,5,6,7,1,0.965926,2.588190e-01,0,0,9
...,...,...,...,...,...,...,...,...,...
874,22,1,9,0,-0.500000,8.660254e-01,0,0,35
875,9,3,6,0,0.707107,-7.071068e-01,0,0,63
876,17,5,9,1,-0.965926,-2.588190e-01,0,0,97
877,1,4,8,0,0.258819,9.659258e-01,0,0,17


In [34]:
# Cargar el LabelEncoder / LabelBinarizer previo
le = ('/content/drive/MyDrive/CienciaDeDatos/Viajes_lter.joblib')

# Separar variables predictoras (X) y objetivo (y)
X_train = df_train.drop(columns=['label'])
y_train = df_train['label']

X_test = df_test.drop(columns=['label'])
y_test = df_test['label']

In [35]:
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Crear y entrenar modelo SVM de regresión
modelo_svm = SVR(kernel='rbf', C=100, epsilon=0.5)  # 'linear', 'rbf', 'poly'
modelo_svm.fit(X_train, y_train)

# Predicción
y_pred = modelo_svm.predict(X_test)

# Evaluación (métricas de regresión, no de clasificación)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.3f}")

MAE:  13.47
RMSE: 25.20
R²:   0.537


#Modelos de IA Supervisados

In [41]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
import pandas as pd

# Escalamos los datos (SVM y regresión lineal lo necesitan; RandomForest no, pero no le afecta)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelos = {
    # Tus modelos originales
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
    "SVR (rbf)": SVR(kernel="rbf", C=100, epsilon=0.5),
    "SVR (linear)": SVR(kernel="linear", C=100, epsilon=0.5),
    "LinearRegression": LinearRegression(),
    # --- Nuevos modelos ---
    # Red Neuronal (Perceptrón Multicapa)
    "Red Neuronal (MLP)": MLPRegressor(
        hidden_layer_sizes=(128, 64), max_iter=500, random_state=42
    ),
    # Modelo de Boosting (Suele superar a RandomForest)
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=200, learning_rate=0.1, random_state=42
    ),
    # Modelo basado en vecinos cercanos
    "KNN": KNeighborsRegressor(n_neighbors=5),
}
resultados = []

for nombre, modelo in modelos.items():
    modelo.fit(X_train_scaled, y_train)
    y_pred = modelo.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    resultados.append({"Modelo": nombre, "MAE": mae, "RMSE": rmse, "R2": r2})

tabla_resultados = pd.DataFrame(resultados).sort_values("R2", ascending=False)
print(tabla_resultados)

               Modelo        MAE       RMSE        R2
4  Red Neuronal (MLP)  11.125866  18.115418  0.760527
5    GradientBoosting  11.030600  18.608312  0.747318
6                 KNN  11.249829  19.207207  0.730791
0        RandomForest  11.626332  19.711697  0.716464
1           SVR (rbf)  11.199990  20.728147  0.686468
3    LinearRegression  18.360146  28.369259  0.412705
2        SVR (linear)  17.591796  29.256607  0.375391


#Busco Optimizarlos o Mejorlos Para aumentar las metricas

In [46]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.neighbors import KNeighborsRegressor

import numpy as np

import pandas as pd


# Escalamos los datos (SVM y regresión lineal lo necesitan; RandomForest no, pero no le afecta)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

modelos = {
    # Basados en árboles: NO necesitan escalado, van directo
    "RandomForest": RandomForestRegressor(
        n_estimators=300, max_depth=15, min_samples_split=5, random_state=42
    ),
    "GradientBoosting": GradientBoostingRegressor(
        n_estimators=300, learning_rate=0.05, max_depth=5, subsample=0.8, random_state=42
    ),

    # Sensibles a escala: cada uno envuelto en un Pipeline con StandardScaler
    "SVR (rbf)": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", SVR(kernel="rbf", C=10, epsilon=0.1, gamma="scale")),
    ]),
    "SVR (linear)": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", SVR(kernel="linear", C=10, epsilon=0.1)),
    ]),
    "Ridge (Linear)": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", Ridge(alpha=1.0)),
    ]),
    "Red Neuronal (MLP)": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", MLPRegressor(
            hidden_layer_sizes=(128, 64), activation="relu", solver="adam",
            alpha=0.0001, learning_rate_init=0.001, max_iter=1000,
            early_stopping=True, random_state=42,
        )),
    ]),
    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("modelo", KNeighborsRegressor(n_neighbors=7, weights="distance", p=2)),
    ]),
}
resultados = []

for nombre, modelo in modelos.items():
    modelo.fit(X_train_scaled, y_train)
    y_pred = modelo.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    resultados.append({"Modelo": nombre, "MAE": mae, "RMSE": rmse, "R2": r2})

tabla_resultados = pd.DataFrame(resultados).sort_values("R2", ascending=False)
print(tabla_resultados)

               Modelo        MAE       RMSE        R2
1    GradientBoosting  10.234897  18.008780  0.763338
5  Red Neuronal (MLP)  11.075207  18.671612  0.745596
0        RandomForest  11.110809  19.057603  0.734969
6                 KNN  11.911613  20.154086  0.703594
2           SVR (rbf)  12.020917  22.686224  0.624435
4      Ridge (Linear)  18.360041  28.369124  0.412711
3        SVR (linear)  17.604657  29.262132  0.375155


In [49]:
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import Ridge

models_and_params = [
    {
        'name': 'Support Vector Machine (SVR)',
        'estimator': SVR(),
        'param_grid': {
            'regressor__C': [0.1, 1, 10, 100],
            'regressor__gamma': ['scale', 'auto', 0.01, 0.1],
            'regressor__kernel': ['rbf', 'linear'],
            'regressor__epsilon': [0.05, 0.1, 0.5]
        }
    },
    {
        'name': 'Decision Tree Regressor',
        'estimator': DecisionTreeRegressor(random_state=42),
        'param_grid': {
            'regressor__criterion': ['squared_error', 'friedman_mse'],
            'regressor__max_depth': [None, 3, 5, 10],
            'regressor__min_samples_split': [2, 5, 10],
            'regressor__min_samples_leaf': [1, 2, 4]
        }
    },
    {
        'name': 'Random Forest Regressor',
        'estimator': RandomForestRegressor(random_state=42),
        'param_grid': {
            'regressor__n_estimators': [50, 100, 200, 300],
            'regressor__max_depth': [None, 5, 10, 15],
            'regressor__min_samples_split': [2, 5],
            'regressor__min_samples_leaf': [1, 2],
            'regressor__bootstrap': [True, False]
        }
    },
    {
        'name': 'K-Nearest Neighbors (KNN)',
        'estimator': KNeighborsRegressor(),
        'param_grid': {
            'regressor__n_neighbors': [3, 5, 7, 9],
            'regressor__weights': ['uniform', 'distance'],
            'regressor__metric': ['euclidean', 'manhattan', 'minkowski']
        }
    },
    {
        'name': 'Ridge Regression',
        'estimator': Ridge(random_state=42),
        'param_grid': {
            'regressor__alpha': [0.01, 0.1, 1, 10, 100],
            'regressor__solver': ['auto', 'svd', 'cholesky', 'lsqr']
        }
    },
    {
        'name': 'Gradient Boosting Regressor',
        'estimator': GradientBoostingRegressor(random_state=42),
        'param_grid': {
            'regressor__n_estimators': [100, 200, 300],
            'regressor__learning_rate': [0.01, 0.05, 0.1, 0.2],
            'regressor__max_depth': [3, 5, 7],
            'regressor__subsample': [0.8, 1.0],
            'regressor__min_samples_split': [2, 5]
        }
    }
]

resultados = []

for nombre, modelo in modelos.items():
    modelo.fit(X_train_scaled, y_train)
    y_pred = modelo.predict(X_test_scaled)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    resultados.append({"Modelo": nombre, "MAE": mae, "RMSE": rmse, "R2": r2})

tabla_resultados = pd.DataFrame(resultados).sort_values("R2", ascending=False)
print(tabla_resultados)

               Modelo        MAE       RMSE        R2
1    GradientBoosting  10.234897  18.008780  0.763338
5  Red Neuronal (MLP)  11.075207  18.671612  0.745596
0        RandomForest  11.110809  19.057603  0.734969
6                 KNN  11.911613  20.154086  0.703594
2           SVR (rbf)  12.020917  22.686224  0.624435
4      Ridge (Linear)  18.360041  28.369124  0.412711
3        SVR (linear)  17.604657  29.262132  0.375155


In [50]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

resultados = []
mejores_modelos = {}

for item in models_and_params:
    nombre = item['name']
    estimator = item['estimator']
    param_grid = item['param_grid']

    pipeline = Pipeline([
        ('scaler', StandardScaler()),
        ('regressor', estimator)
    ])

    grid = GridSearchCV(
        pipeline,
        param_grid,
        cv=5,
        scoring='r2',
        n_jobs=-1
    )
    grid.fit(X_train, y_train)

    mejor_modelo = grid.best_estimator_
    mejores_modelos[nombre] = mejor_modelo

    y_pred = mejor_modelo.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2_test = r2_score(y_test, y_pred)

    resultados.append({
        'Modelo': nombre,
        'R2 (CV train)': grid.best_score_,
        'R2 (test)': r2_test,
        'MAE (test)': mae,
        'RMSE (test)': rmse,
        'Mejores parámetros': grid.best_params_
    })

    print(f"✔ {nombre} listo — mejor R2 en CV: {grid.best_score_:.3f}")

tabla_final = pd.DataFrame(resultados).sort_values('R2 (test)', ascending=False)
print("\nComparación final de modelos:")
print(tabla_final[['Modelo', 'R2 (CV train)', 'R2 (test)', 'MAE (test)', 'RMSE (test)']])

✔ Support Vector Machine (SVR) listo — mejor R2 en CV: 0.706
✔ Decision Tree Regressor listo — mejor R2 en CV: 0.765
✔ Random Forest Regressor listo — mejor R2 en CV: 0.784
✔ K-Nearest Neighbors (KNN) listo — mejor R2 en CV: 0.767
✔ Ridge Regression listo — mejor R2 en CV: 0.458
✔ Gradient Boosting Regressor listo — mejor R2 en CV: 0.792

Comparación final de modelos:
                         Modelo  R2 (CV train)  R2 (test)  MAE (test)  \
5   Gradient Boosting Regressor       0.792109   0.773447   10.325921   
2       Random Forest Regressor       0.784271   0.748508   10.840717   
1       Decision Tree Regressor       0.765382   0.739178   11.211786   
3     K-Nearest Neighbors (KNN)       0.767446   0.730791   11.249829   
0  Support Vector Machine (SVR)       0.706131   0.686039   11.187266   
4              Ridge Regression       0.458468   0.412760   18.359433   

   RMSE (test)  
5    17.619965  
2    18.564461  
1    18.905658  
3    19.207207  
0    20.742323  
4    28.367925 